# GuppyLM in LM Studio laden

Fortsetzung von `guppylm_lehrnotebook.ipynb`. Dort wurde das Modell trainiert;
hier bringen wir es in eine Form, die LM Studio ausführen kann.

**Warum das mehr als eine Dateikonvertierung ist**

LM Studio führt Modelle über llama.cpp aus und liest sie im GGUF-Format. llama.cpp
kennt aber keine beliebigen PyTorch-Modelle: Für jede unterstützte Architektur ist
der Rechengraph fest im C++-Code hinterlegt — `llama`, `gpt2`, `gptneox`, `falcon`
und so weiter. Eine Architektur `guppylm` gibt es dort nicht und wird es nie geben.

Ein Modell weiterzugeben heißt deshalb: **es als eine bereits unterstützte
Architektur ausdrücken**. Das ist die eigentliche Lektion dieses Notebooks.
Zum Glück ist GuppyLM fast wörtlich GPT-2 — gelernte Positions-Embeddings,
Pre-Norm mit LayerNorm, fusioniertes QKV mit Bias, gebundene Ausgabegewichte,
Byte-Level-BPE. Es bleiben drei Unterschiede, und jeder davon ist eine eigene
Fehlerquelle:

| Unterschied | Umgang |
|---|---|
| GPT-2 speichert lineare Schichten transponiert (`Conv1D`) | beim Export transponieren |
| llama.cpp rechnet für `gpt2` **fest GELU** | im Lehrnotebook bereits GELU |
| der Konverter schreibt `n_ff` **fest als 4 × `d_model`** | `ffn_hidden` muss 1536 sein |

*Herkunft:* Gehört zum Lehrmaterial auf Basis von
[arman-bd/guppylm](https://github.com/arman-bd/guppylm/tree/<SHA>) (MIT laut README).
Der Exportweg nach GPT-2/GGUF ist eine Ergänzung und im Original nicht enthalten.
Verwendet [llama.cpp](https://github.com/ggml-org/llama.cpp) (MIT), das geklont,
aber nicht mitverteilt wird. Einzelheiten in `HERKUNFT.md`.

## 1. Voraussetzungen

Neben den Paketen brauchen wir llama.cpp als Quelltext, denn das Konvertierskript
liegt dort. Gebaut werden muss es dafür nicht.

In [ ]:
%pip install -q torch transformers safetensors tokenizers gguf

In [ ]:
import json, os, re, shutil, subprocess, sys
from dataclasses import dataclass
from pathlib import Path

import torch

WORK = Path.cwd() / "guppy"          # identisch zu Notebook 1
EXPORT = WORK / "hf_export"
LLAMA_CPP = Path.cwd() / "llama.cpp"
GGUF_DATEI = WORK / "guppylm-f16.gguf"

if not LLAMA_CPP.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_CPP)],
                   check=True)
print("llama.cpp:", LLAMA_CPP)

for pfad in [WORK / "checkpoints" / "bestes_modell.pt", WORK / "data" / "tokenizer.json"]:
    print(f"  {'gefunden ' if pfad.exists() else 'FEHLT    '} {pfad}")

## 2. Die Modellklassen

Notebook 1 schreibt bewusst keine Python-Dateien, deshalb sind die Klassen hier
noch einmal aufgeführt — unverändert. Wenn Sie dort etwas an der Architektur
ändern, muss es auch hier geändert werden. Genau diese Doppelung ist der Preis
für ein Notebook ohne Projektdateien; ab hier wäre ein echtes Modul die bessere
Wahl.

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GuppyConfig:
    vocab_size: int = 4096
    max_seq_len: int = 128
    d_model: int = 384
    n_layers: int = 6
    n_heads: int = 6
    ffn_hidden: int = 1536
    dropout: float = 0.1
    pad_id: int = 0
    bos_id: int = 1
    eos_id: int = 2


def kausale_maske(T, device=None):
    return torch.tril(torch.ones(T, T, device=device)).view(1, 1, T, T)


class Attention(nn.Module):
    def __init__(self, konfig):
        super().__init__()
        self.n_heads = konfig.n_heads
        self.head_dim = konfig.d_model // konfig.n_heads
        self.qkv = nn.Linear(konfig.d_model, 3 * konfig.d_model)
        self.out = nn.Linear(konfig.d_model, konfig.d_model)
        self.dropout = nn.Dropout(konfig.dropout)

    def forward(self, x, maske=None, return_attn=False):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        roh = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if maske is not None:
            roh = roh.masked_fill(maske == 0, float("-inf"))
        gewichte = F.softmax(roh, dim=-1)
        y = (self.dropout(gewichte) @ v).transpose(1, 2).contiguous().view(B, T, C)
        y = self.out(y)
        return (y, gewichte) if return_attn else y


class FFN(nn.Module):
    def __init__(self, konfig):
        super().__init__()
        self.hoch = nn.Linear(konfig.d_model, konfig.ffn_hidden)
        self.runter = nn.Linear(konfig.ffn_hidden, konfig.d_model)
        self.dropout = nn.Dropout(konfig.dropout)

    def forward(self, x):
        return self.dropout(self.runter(F.gelu(self.hoch(x), approximate="tanh")))


class Block(nn.Module):
    def __init__(self, konfig):
        super().__init__()
        self.norm1 = nn.LayerNorm(konfig.d_model)
        self.attn = Attention(konfig)
        self.norm2 = nn.LayerNorm(konfig.d_model)
        self.ffn = FFN(konfig)

    def forward(self, x, maske=None):
        x = x + self.attn(self.norm1(x), maske)
        x = x + self.ffn(self.norm2(x))
        return x


class GuppyLM(nn.Module):
    def __init__(self, konfig):
        super().__init__()
        self.konfig = konfig
        self.tok_emb = nn.Embedding(konfig.vocab_size, konfig.d_model)
        self.pos_emb = nn.Embedding(konfig.max_seq_len, konfig.d_model)
        self.drop = nn.Dropout(konfig.dropout)
        self.blocks = nn.ModuleList([Block(konfig) for _ in range(konfig.n_layers)])
        self.norm = nn.LayerNorm(konfig.d_model)
        self.lm_head = nn.Linear(konfig.d_model, konfig.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

    def forward(self, idx, ziele=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        maske = kausale_maske(T, idx.device)
        for block in self.blocks:
            x = block(x, maske)
        return self.lm_head(self.norm(x)), None

In [ ]:
from tokenizers import Tokenizer

ckpt = torch.load(WORK / "checkpoints" / "bestes_modell.pt",
                  map_location="cpu", weights_only=False)
konfig = GuppyConfig(**ckpt["konfig"])
modell = GuppyLM(konfig)
modell.load_state_dict(ckpt["model_state_dict"])
modell.eval()

tokenizer = Tokenizer.from_file(str(WORK / "data" / "tokenizer.json"))

print(f"Checkpoint aus Schritt {ckpt['schritt']}")
print(f"{sum(p.numel() for p in modell.parameters()):,} Parameter")

faktor = konfig.ffn_hidden / konfig.d_model
if faktor != 4:
    print(f"\nACHTUNG: ffn_hidden ist {faktor:.0f} x d_model.")
    print("Der gpt2-Konverter schreibt n_ff aber immer als 4 x d_model in die")
    print("GGUF-Datei. llama.cpp erwartet dann Tensoren, die es nicht gibt, und")
    print("verweigert das Laden. Das Modell muss mit ffn_hidden = 4 * d_model")
    print("neu trainiert werden.")
else:
    print("ffn_hidden = 4 x d_model — passt zum gpt2-Konverter.")

## 3. Die Abbildung auf GPT-2

Jeder Tensor bekommt einen neuen Namen. Die Gewichtsmatrizen der linearen
Schichten werden zusätzlich **transponiert**, weil GPT-2 sie als `Conv1D` in der
Form `(ein, aus)` ablegt, `nn.Linear` dagegen als `(aus, ein)`.

Nicht transponiert werden Embeddings, LayerNorm-Parameter und alle Biases.
`lm_head` taucht gar nicht auf: Es teilt sich die Matrix mit `wte`, und
`transformers` stellt diese Bindung beim Bauen des Modells selbst her.

Ein Detail, das leicht schiefgehen könnte und es hier nicht tut: Unser
`reshape(B, T, 3, n_heads, head_dim)` legt Q, K und V in genau derselben
Reihenfolge im Speicher ab wie GPT-2s `c_attn`. Eine Umsortierung ist deshalb
nicht nötig — die Verifikation im nächsten Abschnitt beweist es.

In [ ]:
def nach_gpt2(sd, n_layers):
    neu = {
        "transformer.wte.weight":  sd["tok_emb.weight"],
        "transformer.wpe.weight":  sd["pos_emb.weight"],
        "transformer.ln_f.weight": sd["norm.weight"],
        "transformer.ln_f.bias":   sd["norm.bias"],
    }
    for i in range(n_layers):
        q, g = f"blocks.{i}.", f"transformer.h.{i}."
        neu[g + "ln_1.weight"] = sd[q + "norm1.weight"]
        neu[g + "ln_1.bias"]   = sd[q + "norm1.bias"]
        neu[g + "ln_2.weight"] = sd[q + "norm2.weight"]
        neu[g + "ln_2.bias"]   = sd[q + "norm2.bias"]
        # .T — Conv1D speichert (ein, aus), nn.Linear speichert (aus, ein)
        neu[g + "attn.c_attn.weight"] = sd[q + "attn.qkv.weight"].T.contiguous()
        neu[g + "attn.c_attn.bias"]   = sd[q + "attn.qkv.bias"]
        neu[g + "attn.c_proj.weight"] = sd[q + "attn.out.weight"].T.contiguous()
        neu[g + "attn.c_proj.bias"]   = sd[q + "attn.out.bias"]
        neu[g + "mlp.c_fc.weight"]    = sd[q + "ffn.hoch.weight"].T.contiguous()
        neu[g + "mlp.c_fc.bias"]      = sd[q + "ffn.hoch.bias"]
        neu[g + "mlp.c_proj.weight"]  = sd[q + "ffn.runter.weight"].T.contiguous()
        neu[g + "mlp.c_proj.bias"]    = sd[q + "ffn.runter.bias"]
    return neu


print(f"{len(nach_gpt2(modell.state_dict(), konfig.n_layers))} Tensoren abgebildet")

## 4. Verifikation

Diese Zelle ist der Kern des Notebooks. Sie baut ein echtes `GPT2LMHeadModel`,
lädt die umbenannten Gewichte hinein und vergleicht die Logits mit dem Original.

Stimmen beide auf etwa `1e-6` überein, ist die Abbildung korrekt. Wäre eine
Transposition vergessen oder ein Name vertauscht, würde das Modell trotzdem laden
und trotzdem Text erzeugen — nur schlechteren. Solche Fehler findet man **nur**
durch einen Zahlenvergleich, nie durch Ausprobieren im Chat.

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel

hf_konfig = GPT2Config(
    vocab_size=konfig.vocab_size,
    n_positions=konfig.max_seq_len,
    n_ctx=konfig.max_seq_len,          # der gpt2-Konverter liest genau diesen Schluessel
    n_embd=konfig.d_model,
    n_layer=konfig.n_layers,
    n_head=konfig.n_heads,
    n_inner=konfig.ffn_hidden,
    activation_function="gelu_new",    # = GELU in tanh-Naeherung
    resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0,
    layer_norm_epsilon=1e-5,
    bos_token_id=konfig.bos_id,
    eos_token_id=konfig.eos_id,
    tie_word_embeddings=True,
    architectures=["GPT2LMHeadModel"],
)

hf_modell = GPT2LMHeadModel(hf_konfig)
fehlend, unerwartet = hf_modell.load_state_dict(
    nach_gpt2(modell.state_dict(), konfig.n_layers), strict=False)
hf_modell.eval()

# lm_head.weight darf fehlen — es ist an wte gebunden
assert unerwartet == [], f"unerwartete Tensoren: {unerwartet}"
assert hf_modell.lm_head.weight is hf_modell.transformer.wte.weight

torch.manual_seed(0)
probe = torch.randint(0, konfig.vocab_size, (4, 32))
with torch.no_grad():
    original = modell(probe)[0]
    portiert = hf_modell(probe).logits

abweichung = (original - portiert).abs().max().item()
print(f"maximale Abweichung der Logits: {abweichung:.3e}")
print(f"gleiche Vorhersage an jeder Position: "
      f"{(original.argmax(-1) == portiert.argmax(-1)).all().item()}")

assert abweichung < 1e-4, "Abbildung fehlerhaft — nicht weitermachen!"
print("\nAbbildung verifiziert.")

## 5. Den HuggingFace-Ordner schreiben

Vier Dateien werden gebraucht. Die wichtigste ist unscheinbar:
`tokenizer_config.json` enthält die **Chat-Vorlage**. LM Studio baut daraus die
Prompts, und `eos_token` sagt dem Programm, wann eine Antwort zu Ende ist. Fehlt
das, redet das Modell über das Ende hinaus weiter — ein Fehlerbild, das man leicht
dem Training anlastet, obwohl es an dieser Datei liegt.

In [ ]:
shutil.rmtree(EXPORT, ignore_errors=True)
EXPORT.mkdir(parents=True)

hf_modell.save_pretrained(EXPORT, safe_serialization=True)

# save_pretrained laesst n_ctx weg — der gpt2-Konverter braucht es aber
inhalt = json.loads((EXPORT / "config.json").read_text())
inhalt["n_ctx"] = konfig.max_seq_len
(EXPORT / "config.json").write_text(json.dumps(inhalt, indent=2))

shutil.copy(WORK / "data" / "tokenizer.json", EXPORT / "tokenizer.json")

CHAT_VORLAGE = (
    "{% for m in messages %}"
    "{{ '<|im_start|>' + m['role'] + '\n' + m['content'] + '<|im_end|>' + '\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)

(EXPORT / "tokenizer_config.json").write_text(json.dumps({
    "tokenizer_class": "PreTrainedTokenizerFast",
    "model_max_length": konfig.max_seq_len,
    "bos_token": "<|im_start|>",
    "eos_token": "<|im_end|>",
    "pad_token": "<pad>",
    "clean_up_tokenization_spaces": False,
    "chat_template": CHAT_VORLAGE,
}, indent=2, ensure_ascii=False))

(EXPORT / "special_tokens_map.json").write_text(json.dumps({
    "bos_token": "<|im_start|>",
    "eos_token": "<|im_end|>",
    "pad_token": "<pad>",
}, indent=2))

for p in sorted(EXPORT.iterdir()):
    print(f"  {p.name:28s} {p.stat().st_size/1e6:8.2f} MB")

## 6. Konvertierung nach GGUF

Ein Stolperstein wartet noch. Das Konvertierskript erkennt Tokenizer an einem
**Fingerabdruck**: Es tokenisiert einen festen Testtext und bildet den SHA-256 der
Ausgabe. Nur bekannte Fingerabdrücke werden akzeptiert. Unser Tokenizer wurde auf
Fischsätzen trainiert und ist naturgemäß unbekannt — die Konvertierung bricht mit
`NotImplementedError: BPE pre-tokenizer was not recognized` ab.

Das ist kein Defekt, sondern Absicht: llama.cpp muss wissen, welche
Vorzerlegungsregeln es nachbilden soll, und rät lieber nicht. Da unser Tokenizer
ein gewöhnlicher Byte-Level-BPE ist — genau wie der von GPT-2 — dürfen wir den
Fingerabdruck als `gpt-2` eintragen.

Die nächste Zelle versucht die Konvertierung und sagt Ihnen im Fehlerfall genau,
welche Zeilen wo einzufügen sind. Geändert wird nichts ohne Ihr Zutun.

In [ ]:
def konvertieren():
    umgebung = dict(os.environ, PYTHONPATH=str(LLAMA_CPP / "gguf-py"))
    return subprocess.run(
        [sys.executable, str(LLAMA_CPP / "convert_hf_to_gguf.py"), str(EXPORT),
         "--outfile", str(GGUF_DATEI), "--outtype", "f16"],
        capture_output=True, text=True, env=umgebung)


def vokabeldatei():
    """get_vocab_base_pre() liegt je nach llama.cpp-Version woanders."""
    for kandidat in [LLAMA_CPP / "conversion" / "base.py",
                     LLAMA_CPP / "convert_hf_to_gguf.py"]:
        if kandidat.exists() and "def get_vocab_base_pre" in kandidat.read_text():
            return kandidat
    raise FileNotFoundError("get_vocab_base_pre() nicht gefunden")


ergebnis = konvertieren()
ausgabe = ergebnis.stdout + ergebnis.stderr

if ergebnis.returncode == 0:
    print(f"GGUF geschrieben: {GGUF_DATEI} "
          f"({GGUF_DATEI.stat().st_size/1e6:.1f} MB)")
elif "pre-tokenizer was not recognized" in ausgabe:
    FINGERABDRUCK = re.search(r"chkhsh:\s+([0-9a-f]{64})", ausgabe).group(1)
    print("Der Tokenizer ist llama.cpp unbekannt. Fingerabdruck:")
    print(f"\n    {FINGERABDRUCK}\n")
    print(f"Einzutragen in {vokabeldatei()}, in get_vocab_base_pre(),")
    print("unmittelbar vor der Zeile 'if res is None:':\n")
    print(f'    if chkhsh == "{FINGERABDRUCK}":')
    print('        # eigener ByteLevel-BPE-Tokenizer (GuppyLM)')
    print('        res = "gpt-2"')
    print("\nEntweder von Hand eintragen oder die naechste Zelle benutzen.")
else:
    print(ausgabe[-2000:])

Die folgende Zelle nimmt Ihnen das Eintragen ab. Sie tut das **nur**, wenn Sie
`PATCH_ERLAUBT = True` setzen — Werkzeuge sollen beraten, nicht unbemerkt
verändern, und hier wird eine fremde Quelldatei angefasst.

In [ ]:
PATCH_ERLAUBT = True      # bewusst auf True setzen

if ergebnis.returncode == 0:
    print("Konvertierung war bereits erfolgreich — nichts zu tun.")
elif not PATCH_ERLAUBT:
    print("PATCH_ERLAUBT ist False. Es wurde nichts geaendert.")
else:
    datei = vokabeldatei()
    quelle = datei.read_text()
    anker = "        if res is None:"
    start = quelle.index("def get_vocab_base_pre")
    stelle = quelle.index(anker, start)

    einschub = (f'        if chkhsh == "{FINGERABDRUCK}":\n'
                f'            # eigener ByteLevel-BPE-Tokenizer (GuppyLM)\n'
                f'            res = "gpt-2"\n\n')

    if FINGERABDRUCK in quelle:
        print("Fingerabdruck steht bereits in der Datei.")
    else:
        datei.write_text(quelle[:stelle] + einschub + quelle[stelle:])
        print(f"{datei} ergaenzt.")

    ergebnis = konvertieren()
    if ergebnis.returncode == 0:
        print(f"\nGGUF geschrieben: {GGUF_DATEI} "
              f"({GGUF_DATEI.stat().st_size/1e6:.1f} MB)")
    else:
        print((ergebnis.stdout + ergebnis.stderr)[-2000:])

## 7. Die GGUF-Datei gegenlesen

Vor dem Kopieren lohnt ein Blick in die Kopfdaten. Besonders auf
`feed_forward_length`: Steht dort nicht das Vierfache von `embedding_length`,
passt die Datei nicht zu ihren eigenen Tensoren und llama.cpp wird sie ablehnen.

In [ ]:
sys.path.insert(0, str(LLAMA_CPP / "gguf-py"))
from gguf import GGUFReader

leser = GGUFReader(str(GGUF_DATEI))
interessant = ["general.architecture", "gpt2.context_length",
               "gpt2.embedding_length", "gpt2.feed_forward_length",
               "gpt2.block_count", "gpt2.attention.head_count",
               "tokenizer.ggml.model", "tokenizer.ggml.eos_token_id"]

for name in interessant:
    feld = leser.fields.get(name)
    if feld is not None:
        print(f"  {name:36s} {feld.contents()}")

form = next(t.shape for t in leser.tensors if t.name == "blk.0.ffn_up.weight")
print(f"\n  blk.0.ffn_up.weight               {tuple(int(x) for x in form)}")
print(f"  Tensoren insgesamt                {len(leser.tensors)}")
print(f"  Chat-Vorlage vorhanden            "
      f"{'tokenizer.chat_template' in leser.fields}")

## 8. In LM Studio einbinden

LM Studio erwartet die Struktur `<verlag>/<modellname>/datei.gguf` unterhalb
seines Modellordners. Wo dieser liegt, zeigt die App unter **My Models**; die
Zelle sucht die üblichen Orte ab.

In [ ]:
kandidaten = [Path.home() / ".lmstudio" / "models",
              Path.home() / ".cache" / "lm-studio" / "models"]
modellordner = next((k for k in kandidaten if k.exists()), None)

VERLAG, MODELLNAME = "iludis", "guppylm-12M"

if modellordner is None:
    print("Kein LM-Studio-Modellordner gefunden. Gesucht wurde in:")
    for k in kandidaten:
        print(f"  {k}")
    print("\nDen tatsaechlichen Pfad zeigt LM Studio unter 'My Models'.")
    print(f"Dorthin gehoert: <ordner>/{VERLAG}/{MODELLNAME}/{GGUF_DATEI.name}")
else:
    ziel = modellordner / VERLAG / MODELLNAME
    ziel.mkdir(parents=True, exist_ok=True)
    shutil.copy(GGUF_DATEI, ziel / GGUF_DATEI.name)
    print(f"kopiert nach {ziel / GGUF_DATEI.name}")
    print("\nIn LM Studio unter 'My Models' neu einlesen lassen.")

## 9. Einstellungen in LM Studio

Die Voreinstellungen der App sind auf Milliardenmodelle abgestimmt und passen
hier nicht. Drei Punkte lohnen die Kontrolle:

- **Wiederholungsstrafe auf 1,0.** Der übliche Wert 1,1 bestraft bei einem
  Vokabular von 4096 genau die Wörter, aus denen die Fischsprache besteht.
- **Kontextlänge 128.** Mehr kann das Modell nicht; die Positions-Embeddings
  existieren für längere Sequenzen schlicht nicht.
- **Kein System-Prompt.** Trainiert wurde nur auf `user`/`assistant`; eine
  Systemrolle hat das Modell nie gesehen.

Und die Erwartung: Es antwortet in kurzen, kleingeschriebenen Fischsätzen über
Wasser, Futter und Licht. Mehr steckt in 12 Millionen Parametern und 60 Themen
nicht drin. Dass es überhaupt in derselben Anwendung läuft wie ein Modell mit
tausendfacher Größe, ist der Punkt der Übung.

### Gegenprobe ohne LM Studio

Wer llama.cpp ohnehin gebaut hat, kann die Konvertierung direkt prüfen. Bei
Temperatur 0 muss dieselbe Antwort herauskommen wie im Lehrnotebook bei
`temperatur` nahe 0:

```
cmake -B build llama.cpp && cmake --build build -j
./build/bin/llama-cli -m guppy/guppylm-f16.gguf -p "hallo guppy" --temp 0 -n 32
```

Weichen die Ausgaben deutlich voneinander ab, obwohl die Verifikation in
Abschnitt 4 bestanden wurde, liegt der Fehler zwischen GGUF und llama.cpp — dann
sind die Kopfdaten aus Abschnitt 7 die erste Anlaufstelle.